<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Ética, IA/ML/DL y el Paradigma Generativo ⚖️🧬 🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.1em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial — Taller Práctico Evaluativo
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 03 • Para Dummies
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

---
## 🎈 ¿Qué vamos a hacer en este taller?

1. ⚖️ **Cazar un sesgo escondido** en una base de datos de créditos bancarios (¡ya sabemos que existe, tú lo vas a medir!).
2. 🧠 **Armar, pieza por pieza, una neurona artificial** usando solo NumPy.
3. ✍️ **Opinar sobre los riesgos** de una herramienta de IA Generativa que ya uses o conozcas.

> ⚠️ **Instrucciones de entrega:** completa cada celda marcada con `# TODO:`. No se aceptan notebooks con celdas sin ejecutar o con errores. Antes de entregar, usa *Restart Kernel and Run All Cells* para verificar que todo el notebook corre de principio a fin.

---
### 📦 Los Datos: Solicitudes de Crédito (ya resuelto, no lo modifiques)

Piensa en esto como una hoja de Excel con 600 solicitudes de crédito ficticias. A propósito, le "escondimos" un sesgo: a igualdad de ingresos y puntaje crediticio, al grupo `"Mujer"` le baja artificialmente la probabilidad de que el crédito se apruebe. Tu misión en el Reto 1 será descubrirlo con números.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 600

genero = rng.choice(["Hombre", "Mujer"], size=n, p=[0.5, 0.5])
ingresos = rng.normal(3_000_000, 800_000, size=n).clip(500_000)
puntaje_crediticio = rng.normal(650, 80, size=n).clip(300, 850)

# Sesgo deliberado: a igualdad de ingresos y puntaje, "Mujer" tiene una
# probabilidad de aprobación artificialmente menor (sesgo introducido a propósito
# para que lo detectes en el Ejercicio 1.1).
prob_base = 1 / (1 + np.exp(-(0.0000015 * (ingresos - 3_000_000) + 0.01 * (puntaje_crediticio - 650))))
penalizacion_sesgo = np.where(genero == "Mujer", 0.22, 0.0)
prob_aprobacion = np.clip(prob_base - penalizacion_sesgo, 0.02, 0.98)

aprobado = rng.binomial(1, prob_aprobacion)

df_creditos = pd.DataFrame({
    "genero": genero,
    "ingresos": ingresos.round(0),
    "puntaje_crediticio": puntaje_crediticio.round(0),
    "aprobado": aprobado,
})
display(df_creditos.head())

---
### 📌 Reto 1: ¡Encuentra el Sesgo Escondido! 🧸

**Ejercicio 1.1:**
1. Calcula qué porcentaje de hombres fue aprobado y qué porcentaje de mujeres fue aprobado (pista: `df_creditos.groupby('genero')['aprobado'].mean()`).
2. Divide la tasa más baja entre la tasa más alta (eso se llama *disparate impact ratio*). Si el resultado es menor a 0.8, ¡hay señales de sesgo!
3. Escribe qué concluyes en `conclusion_sesgo`.

In [ ]:
# TODO paso 1: calcula la tasa de aprobación por género
tasa_aprobacion_por_genero = None  # Pista: df_creditos.groupby('genero')['aprobado'].mean()

# TODO paso 2: calcula tasa_menor / tasa_mayor entre los dos grupos
disparate_impact_ratio = None

print("Tasa de aprobación por género:\n", tasa_aprobacion_por_genero)
print("\nDisparate impact ratio:", disparate_impact_ratio)

conclusion_sesgo = None  # TODO paso 3: ¿hay sesgo? ¿qué tan grave es según la regla del 80%?
print("\nConclusión:", conclusion_sesgo)

---
### 📌 Reto 2: Arma tu Primera "Neurona" con NumPy 🧸🧠

Una capa de red neuronal en el fondo solo hace 2 cosas: (1) una multiplicación de matrices + suma, y (2) le aplica una función que "decide" qué tanto se activa.

**Ejemplo ya resuelto — función sigmoide:**
```python
def sigmoide(z):
    return 1 / (1 + np.exp(-z))
```

**Ejercicio 2.1:** completa `relu` (mucho más simple: si el número es negativo se vuelve 0, si es positivo se queda igual) y completa `forward_capa_densa`.

In [ ]:
import numpy as np

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    # TODO: si z es negativo, el resultado es 0; si es positivo, el resultado es z.
    # Pista: np.maximum(0, z) lo hace elemento a elemento en un array completo.
    pass

def forward_capa_densa(X, W, b, activacion="relu"):
    """X: entradas, W: pesos, b: sesgos, activacion: "relu" o "sigmoide" """
    z = None  # TODO: calcula z = X @ W + b (el "@" es multiplicación de matrices)
    # TODO: si activacion == "relu", aplica relu(z); si no, aplica sigmoide(z)
    pass


X_prueba = np.array([[1.0, 2.0, -1.0], [0.5, -0.5, 2.0]])
W_prueba = np.random.default_rng(0).normal(size=(3, 4))
b_prueba = np.zeros(4)

salida = forward_capa_densa(X_prueba, W_prueba, b_prueba, activacion="relu")
print("Salida de la capa densa:\n", salida)

---
### 📌 Reto 3: Tu Opinión sobre una Herramienta de IA Generativa 🧸

**Ejercicio 3.1:** Elige una herramienta que ya conozcas (ChatGPT, Gemini, Midjourney, Copilot, etc.) y responde (mínimo 100 palabras) apoyándote en estas preguntas guía:
- ¿Qué riesgo ético le ves (sesgo, desinformación, derechos de autor, empleos, privacidad, "alucinaciones")?
- Cuenta un ejemplo real o inventado donde ese riesgo se note.
- ¿Qué recomendarías para usarla de forma más responsable?

#### ✍️ Tu respuesta

_Herramienta elegida: ..._

_Responde aquí las 3 preguntas guía (mínimo 100 palabras). Reemplaza este texto por tu contenido._

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial</i><br>
    Taller Para Dummies — Módulo 03: Ética y el Nuevo Paradigma de la IA
  </p>
</div>